# This code requires GPU power and so was run on Google Colab. 

In [ ]:
# First, mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

# Now load your CSV with the correct path
import pandas as pd
df_combined = pd.read_csv("/content/drive/My Drive/Colab Notebooks/names_train_combined_cleaned.csv")
df_combined.info()

In [ ]:
df_combined.head()

In [ ]:
import sys
print(sys.executable)
print(sys.version)

/usr/bin/python3
3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [ ]:
# If in Jupyter, run this cell
import subprocess
import sys

# subprocess.check_call([sys.executable, "-m", "pip", "install", "torch"])
# subprocess.check_call([sys.executable, "-m", "pip", "install", "transformers"])
# subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn"])

0

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.device_count())

# This works even without nvidia-smi
import subprocess
subprocess.run(["wmic", "path", "win32_VideoController", "get", "name"], shell=True)

True
1


CompletedProcess(args=['wmic', 'path', 'win32_VideoController', 'get', 'name'], returncode=127)

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Copy the file from Drive to local Colab environment
!cp "/content/drive/My Drive/Colab Notebooks/names_train_combined_cleaned.csv" "names_train_combined_cleaned.csv"

# Now you can read it with just the filename
df = pd.read_csv("names_train_combined_cleaned.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_7056/2652034678.py:9: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("names_train_combined_cleaned.csv")


In [ ]:
import os, json, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

def seed_everything(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

seed_everything()

CFG = dict(
    model_name   = "google/muril-base-cased",
    csv_path     = "names_train_combined_cleaned.csv",
    max_len      = 64,
    batch_size   = 64,
    epochs       = 3,
    lr           = 2e-5,
    warmup_ratio = 0.1,
    val_size     = 0.15,
    seed         = 42,
    save_dir     = "muril_per_religion",
    device       = "cuda" if torch.cuda.is_available() else "cpu",
)
os.makedirs(CFG["save_dir"], exist_ok=True)
print(f"Device: {CFG['device']}")

Device: cuda


In [ ]:
R_LABELS = ["H", "M", "O", "U"]

SUBMODEL_VOCAB = {
    "H": {
        "G":    ["GEN", "SC", "ST", "OBC", "U"],
        "SG":   ["GUC", "MAT", "RAJ", "NAM", "MAH", "SAN", "TEA", "O", "U"],
        "CONF": ["H", "M", "L"],
    },
    "M": {
        "G":    ["GEN", "U"],
        "SG":   ["BM", "UM", "O", "U"],
        "CONF": ["H", "M", "L"],
    },
    "O": {
        # Christians, Sikhs, Jains, Buddhists etc.
        # G can be anything; SG is just O or U since we have no finer taxonomy
        "G":    ["GEN", "SC", "ST", "OBC", "U"],
        "SG":   ["O", "U"],
        "CONF": ["H", "M", "L"],
    },
}
# R=U rows still get G=U, SG=U, CONF=L — no model trained for pure unknowns

with open(os.path.join(CFG["save_dir"], "vocab.json"), "w") as f:
    json.dump({"R": R_LABELS, "submodels": SUBMODEL_VOCAB}, f, indent=2)

for r, v in SUBMODEL_VOCAB.items():
    print(f"R={r}  G:{v['G']}  SG:{v['SG']}")

R=H  G:['GEN', 'SC', 'ST', 'OBC', 'U']  SG:['GUC', 'MAT', 'RAJ', 'NAM', 'MAH', 'SAN', 'TEA', 'O', 'U']
R=M  G:['GEN', 'U']  SG:['BM', 'UM', 'O', 'U']
R=O  G:['GEN', 'SC', 'ST', 'OBC', 'U']  SG:['O', 'U']


In [ ]:
# df = pd.read_csv(CFG["csv_path"])
df = df.dropna(subset=["R","G","SG","CONF"]).copy()
print(f"Labelled rows: {len(df):,}")

for col in ["R","G","SG","CONF"]:
    df[col] = df[col].str.strip()

# Collapse anything that isn't H/M/U into O
valid_R = {"H", "M", "U"}
df["R"] = df["R"].apply(lambda x: x if x in valid_R else "O")
print("\nR distribution after remapping:")
print(df["R"].value_counts().to_string())

# Input text: name + district
df["text"] = df["name"].str.strip() + " [SEP] " + df["district"].str.strip()

# Encode R for stage-1 model
df["R_enc"] = df["R"].apply(lambda x: R_LABELS.index(x))  # all values now guaranteed in R_LABELS

# Encode G/SG/CONF per religion sub-model
for r_val, vocab in SUBMODEL_VOCAB.items():
    mask = df["R"] == r_val
    sub  = df[mask].copy()
    for col, labels in vocab.items():
        df.loc[mask, f"{col}_enc_{r_val}"] = sub[col].apply(
            lambda x: labels.index(x) if x in labels else labels.index("U")
        )

# 85/15 stratified split on R
train_df, val_df = train_test_split(
    df, test_size=CFG["val_size"], random_state=CFG["seed"], stratify=df["R_enc"]
)
print(f"\nTrain: {len(train_df):,}  |  Val: {len(val_df):,}")
print("\nR distribution (train):")
print(train_df["R"].value_counts().to_string())

Labelled rows: 682,008

R distribution after remapping:
R
H    461179
M    217401
U      2091
O      1337

Train: 579,706  |  Val: 102,302

R distribution (train):
R
H    392002
M    184791
U      1777
O      1136


In [ ]:
class VoterDataset(Dataset):
    """
    r_val=None  → Stage-1 dataset (all rows, label = R)
    r_val='H'   → Hindu sub-model dataset (only H rows, labels = G/SG/CONF)
    r_val='M'   → Muslim sub-model dataset (only M rows, labels = G/SG/CONF)
    """
    def __init__(self, df, tokenizer, max_len, r_val=None):
        if r_val is not None:
            df = df[df["R"] == r_val].copy()

        self.texts     = df["text"].tolist()
        self.r_val     = r_val
        self.tokenizer = tokenizer
        self.max_len   = max_len

        if r_val is None:
            # Stage 1: predict R
            self.labels = {"R": torch.tensor(df["R_enc"].values, dtype=torch.long)}
        else:
            # Sub-model: predict G, SG, CONF for this religion
            self.labels = {
                col: torch.tensor(df[f"{col}_enc_{r_val}"].values, dtype=torch.long)
                for col in ["G", "SG", "CONF"]
            }

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len, padding="max_length",
            truncation=True, return_tensors="pt",
        )
        item = {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "token_type_ids": enc.get(
                "token_type_ids", torch.zeros(self.max_len, dtype=torch.long)
            ).squeeze(0),
        }
        item.update({k: v[idx] for k, v in self.labels.items()})
        return item

In [ ]:
class MuRILClassifier(nn.Module):
    """
    A single MuRIL encoder with one linear head per output label.
    Instantiated once for Stage 1 (R only) and once per sub-model (G+SG+CONF).
    """
    def __init__(self, model_name, head_sizes: dict, dropout=0.1):
        """
        head_sizes: e.g. {"R": 4}  or  {"G": 5, "SG": 9, "CONF": 3}
        """
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        h = self.encoder.config.hidden_size  # 768

        self.drop  = nn.Dropout(dropout)
        self.heads = nn.ModuleDict({
            name: nn.Linear(h, n) for name, n in head_sizes.items()
        })

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out = self.encoder(input_ids=input_ids,
                           attention_mask=attention_mask,
                           token_type_ids=token_type_ids)
        cls = self.drop(out.last_hidden_state[:, 0, :])
        return {name: head(cls) for name, head in self.heads.items()}

In [ ]:
def run_epoch(model, loader, optimizer=None, scheduler=None, device=CFG["device"], train=True):
    model.train() if train else model.eval()
    label_keys = [k for k in next(iter(loader)).keys()
                  if k not in ("input_ids", "attention_mask", "token_type_ids")]

    total_loss, total_correct, total_n = 0, {k: 0 for k in label_keys}, 0
    criterion = nn.CrossEntropyLoss()

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            ids   = batch["input_ids"].to(device)
            mask  = batch["attention_mask"].to(device)
            ttids = batch["token_type_ids"].to(device)
            targets = {k: batch[k].to(device) for k in label_keys}

            logits = model(ids, mask, ttids)
            loss   = sum(criterion(logits[k], targets[k]) for k in label_keys)

            if train:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step()

            bs = ids.size(0); total_loss += loss.item() * bs; total_n += bs
            for k in label_keys:
                total_correct[k] += (logits[k].argmax(-1) == targets[k]).sum().item()

    acc = {k: total_correct[k] / total_n for k in label_keys}
    return total_loss / total_n, acc


def make_optimizer_scheduler(model, n_batches):
    opt  = AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.01)
    total   = n_batches * CFG["epochs"]
    warmup  = int(total * CFG["warmup_ratio"])
    sched   = get_linear_schedule_with_warmup(opt, warmup, total)
    return opt, sched

In [ ]:
def train_model(model, train_loader, val_loader, save_name):
    """Train for CFG['epochs'] epochs, save best checkpoint, return history."""
    opt, sched = make_optimizer_scheduler(model, len(train_loader))
    best_val_loss = float("inf")
    history = []

    print(f"\n── Training {save_name} ──────────────────────────────────────")
    for epoch in range(1, CFG["epochs"] + 1):
        tr_loss, tr_acc = run_epoch(model, train_loader, opt, sched, train=True)
        vl_loss, vl_acc = run_epoch(model, val_loader,             train=False)

        acc_str = "  ".join(f"{k}={v:.3f}" for k,v in vl_acc.items())
        print(f"  Ep {epoch}  tr={tr_loss:.4f} vl={vl_loss:.4f}  [{acc_str}]")
        history.append({"epoch": epoch, "tr_loss": tr_loss, "vl_loss": vl_loss, **{f"val_{k}": v for k,v in vl_acc.items()}})

        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            torch.save(model.state_dict(), os.path.join(CFG["save_dir"], f"{save_name}.pt"))
            print(f"    ✓ saved")

    return history

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"])

# ── Datasets ────────────────────────────────────────────────────────────────
r_train_ds = VoterDataset(train_df, tokenizer, CFG["max_len"], r_val=None)
r_val_ds   = VoterDataset(val_df,   tokenizer, CFG["max_len"], r_val=None)

r_train_loader = DataLoader(r_train_ds, batch_size=CFG["batch_size"],
                             shuffle=True,  num_workers=2, pin_memory=True)
r_val_loader   = DataLoader(r_val_ds,   batch_size=CFG["batch_size"],
                             shuffle=False, num_workers=2, pin_memory=True)

# ── Model ───────────────────────────────────────────────────────────────────
r_model = MuRILClassifier(CFG["model_name"], {"R": len(R_LABELS)}).to(CFG["device"])

r_history = train_model(r_model, r_train_loader, r_val_loader, save_name="stage1_R")

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModel
import os
from torch.utils.data import Dataset, DataLoader

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Load tokenizer
model_dir = "/content/muril_per_religion"
tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")

# CORRECTED: Model class that matches your training architecture
class MuRILClassifier(torch.nn.Module):
    def __init__(self, model_name="google/muril-base-cased", head_sizes=None, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        # If head_sizes not provided, default to R classification
        if head_sizes is None:
            head_sizes = {"R": 4}  # H, M, U, O

        # Create heads dictionary (matching your training)
        self.heads = torch.nn.ModuleDict({
            name: torch.nn.Linear(self.encoder.config.hidden_size, size)
            for name, size in head_sizes.items()
        })
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0, :]  # CLS token
        pooled = self.dropout(pooled)

        # Return dictionary of heads (matching training)
        return {name: head(pooled) for name, head in self.heads.items()}

# Load the saved state dict
model_path = os.path.join(model_dir, "stage1_R.pt")
state_dict = torch.load(model_path, map_location=device)

print("Keys in saved model:")
for i, key in enumerate(list(state_dict.keys())[:10]):
    print(f"  {key}")
print(f"\nTotal keys: {len(state_dict)}")

# Create model with correct architecture
model = MuRILClassifier(head_sizes={"R": 4})

# Load the state dict directly (keys should match now)
try:
    model.load_state_dict(state_dict, strict=True)
    print("✓ Model loaded with strict=True")
except RuntimeError as e:
    print(f"Strict loading failed: {e}")
    print("\nTrying with strict=False...")
    model.load_state_dict(state_dict, strict=False)
    print("✓ Model loaded with strict=False")

model.to(device)
model.eval()

print("\n✓ Model ready for predictions!")

# Dataset for batch prediction
class PredictionDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=64):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
        }

# Batch prediction function
def predict_batch_gpu(df_missing, model, tokenizer, batch_size=256):
    """Fast GPU batch prediction for rows with missing R"""
    if len(df_missing) == 0:
        return [], []

    # Create texts only for missing rows
    texts = [f"{row['name']} [SEP] {row['district']}"
             for _, row in df_missing.iterrows()]

    # Create dataset and dataloader
    dataset = PredictionDataset(texts, tokenizer)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    R_LABELS = ['H', 'M', 'U', 'O']
    predictions = []
    confidences = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids, attention_mask)
            # outputs is a dict with 'R' key
            logits = outputs['R']
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            predictions.extend([R_LABELS[p] for p in preds.cpu().numpy()])
            confidences.extend(probs.max(dim=1)[0].cpu().numpy())

            # Print progress every 10 batches
            if (batch_idx + 1) % 10 == 0:
                processed = (batch_idx + 1) * batch_size
                if processed > len(df_missing):
                    processed = len(df_missing)
                print(f"  Processed {processed:,}/{len(df_missing):,} rows...")

    return predictions, confidences

# Load your CSV
print("\n" + "="*50)
print("LOADING DATA")
print("="*50)
csv_path = "/content/drive/My Drive/Colab Notebooks/names_train_combined_cleaned.csv"  # CHANGE THIS
df = pd.read_csv(csv_path)

# Check current R column status
print(f"\nTotal rows: {len(df):,}")
print(f"Rows with R values: {df['R'].notna().sum():,}")
print(f"Rows missing R (NaN): {df['R'].isna().sum():,}")

# Find missing R rows
missing_mask = df['R'].isna()
df_missing = df[missing_mask].copy()

if len(df_missing) == 0:
    print("\n✓ No missing R values found! All rows have R already.")
else:
    print(f"\n" + "="*50)
    print(f"PREDICTING FOR {len(df_missing):,} MISSING ROWS")
    print("="*50)

    import time
    start = time.time()

    # Run predictions only on missing rows
    preds, confs = predict_batch_gpu(df_missing, model, tokenizer, batch_size=256)

    # Add predictions to the original dataframe ONLY for missing rows
    df.loc[missing_mask, 'R_predicted'] = preds
    df.loc[missing_mask, 'R_confidence'] = confs
    df.loc[missing_mask, 'R'] = preds  # Fill original column with predictions

    elapsed = time.time() - start

    print(f"\n" + "="*50)
    print("RESULTS")
    print("="*50)
    print(f"✓ Completed in {elapsed/60:.1f} minutes")
    print(f"  Speed: {len(df_missing)/elapsed:.1f} predictions/second")

    # Show distribution of predictions
    print(f"\nPredicted R distribution:")
    print(df[missing_mask]['R_predicted'].value_counts())

    # Show confidence statistics
    print(f"\nConfidence scores:")
    print(df[missing_mask]['R_confidence'].describe())

    # Show some examples
    print(f"\nSample predictions (first 10):")
    sample = df[missing_mask][['name', 'district', 'R_predicted', 'R_confidence']].head(10)
    print(sample.to_string())

    # Save results to Google Drive
    output_path = "/content/drive/My Drive/Colab Notebooks/file_with_predictions.csv"
    df.to_csv(output_path, index=False)
    print(f"\n✓ Saved to: {output_path}")

    # Also save only the predictions for missing rows
    missing_output = "/content/drive/My Drive/Colab Notebooks/missing_r_predictions.csv"
    df[missing_mask][['name', 'district', 'R_predicted', 'R_confidence']].to_csv(missing_output, index=False)
    print(f"✓ Predictions for missing rows saved to: {missing_output}")

# Verify final status
print(f"\n" + "="*50)
print("FINAL STATUS")
print("="*50)
print(f"Total rows: {len(df):,}")
print(f"Rows with R values: {df['R'].notna().sum():,}")
print(f"Rows still missing R: {df['R'].isna().sum():,}")

if df['R'].isna().sum() == 0:
    print("\n✅ COMPLETE! All rows now have R values!")

Using device: cuda
GPU: Tesla T4
Memory: 15.6 GB
Keys in saved model:
  encoder.embeddings.word_embeddings.weight
  encoder.embeddings.position_embeddings.weight
  encoder.embeddings.token_type_embeddings.weight
  encoder.embeddings.LayerNorm.weight
  encoder.embeddings.LayerNorm.bias
  encoder.encoder.layer.0.attention.self.query.weight
  encoder.encoder.layer.0.attention.self.query.bias
  encoder.encoder.layer.0.attention.self.key.weight
  encoder.encoder.layer.0.attention.self.key.bias
  encoder.encoder.layer.0.attention.self.value.weight

Total keys: 201


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: google/muril-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded with strict=True

✓ Model ready for predictions!

LOADING DATA


/tmp/ipykernel_7056/1408459902.py:140: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)



Total rows: 1,049,029
Rows with R values: 682,008
Rows missing R (NaN): 367,021

PREDICTING FOR 367,021 MISSING ROWS
  Processed 2,560/367,021 rows...
  Processed 5,120/367,021 rows...
  Processed 7,680/367,021 rows...
  Processed 10,240/367,021 rows...
  Processed 12,800/367,021 rows...
  Processed 15,360/367,021 rows...
  Processed 17,920/367,021 rows...
  Processed 20,480/367,021 rows...
  Processed 23,040/367,021 rows...
  Processed 25,600/367,021 rows...
  Processed 28,160/367,021 rows...
  Processed 30,720/367,021 rows...
  Processed 33,280/367,021 rows...
  Processed 35,840/367,021 rows...
  Processed 38,400/367,021 rows...
  Processed 40,960/367,021 rows...
  Processed 43,520/367,021 rows...
  Processed 46,080/367,021 rows...
  Processed 48,640/367,021 rows...
  Processed 51,200/367,021 rows...
  Processed 53,760/367,021 rows...
  Processed 56,320/367,021 rows...
  Processed 58,880/367,021 rows...
  Processed 61,440/367,021 rows...
  Processed 64,000/367,021 rows...
  Processe

In [ ]:
h_train_ds = VoterDataset(train_df, tokenizer, CFG["max_len"], r_val="H")
h_val_ds   = VoterDataset(val_df,   tokenizer, CFG["max_len"], r_val="H")

print(f"Hindu rows — train: {len(h_train_ds):,}  val: {len(h_val_ds):,}")

h_train_loader = DataLoader(h_train_ds, batch_size=CFG["batch_size"],
                             shuffle=True,  num_workers=2, pin_memory=True)
h_val_loader   = DataLoader(h_val_ds,   batch_size=CFG["batch_size"],
                             shuffle=False, num_workers=2, pin_memory=True)

h_model = MuRILClassifier(
    CFG["model_name"],
    head_sizes={col: len(labels) for col, labels in SUBMODEL_VOCAB["H"].items()}
).to(CFG["device"])

h_history = train_model(h_model, h_train_loader, h_val_loader, save_name="submodel_H")

In [ ]:
m_train_ds = VoterDataset(train_df, tokenizer, CFG["max_len"], r_val="M")
m_val_ds   = VoterDataset(val_df,   tokenizer, CFG["max_len"], r_val="M")

print(f"Muslim rows — train: {len(m_train_ds):,}  val: {len(m_val_ds):,}")

m_train_loader = DataLoader(m_train_ds, batch_size=CFG["batch_size"],
                             shuffle=True,  num_workers=2, pin_memory=True)
m_val_loader   = DataLoader(m_val_ds,   batch_size=CFG["batch_size"],
                             shuffle=False, num_workers=2, pin_memory=True)

m_model = MuRILClassifier(
    CFG["model_name"],
    head_sizes={col: len(labels) for col, labels in SUBMODEL_VOCAB["M"].items()}
).to(CFG["device"])

m_history = train_model(m_model, m_train_loader, m_val_loader, save_name="submodel_M")

In [ ]:
o_train_ds = VoterDataset(train_df, tokenizer, CFG["max_len"], r_val="O")
o_val_ds   = VoterDataset(val_df,   tokenizer, CFG["max_len"], r_val="O")

print(f"Other rows — train: {len(o_train_ds):,}  val: {len(o_val_ds):,}")

# If the O slice is very small (<500 rows), increase dropout to regularise
o_dropout = 0.3 if len(o_train_ds) < 500 else 0.1

o_model = MuRILClassifier(
    CFG["model_name"],
    head_sizes={col: len(labels) for col, labels in SUBMODEL_VOCAB["O"].items()},
    dropout=o_dropout,
).to(CFG["device"])

# Also use a smaller batch size in case the dataset is tiny
o_batch = min(CFG["batch_size"], max(8, len(o_train_ds) // 20))
o_train_loader = DataLoader(o_train_ds, batch_size=o_batch,
                             shuffle=True,  num_workers=4, pin_memory=True)
o_val_loader   = DataLoader(o_val_ds,   batch_size=o_batch,
                             shuffle=False, num_workers=4, pin_memory=True)

o_history = train_model(o_model, o_train_loader, o_val_loader, save_name="submodel_O")

In [ ]:
def load_best(model, name):
    model.load_state_dict(torch.load(
        os.path.join(CFG["save_dir"], f"{name}.pt"), map_location=CFG["device"]
    ))
    return model.eval()

r_model = load_best(r_model, "stage1_R")
h_model = load_best(h_model, "submodel_H")
m_model = load_best(m_model, "submodel_M")
o_model = load_best(o_model, "submodel_O")  # added

def collect_preds(model, loader, label_keys):
    preds = {k: [] for k in label_keys}
    truth = {k: [] for k in label_keys}
    with torch.no_grad():
        for batch in loader:
            ids   = batch["input_ids"].to(CFG["device"])
            mask  = batch["attention_mask"].to(CFG["device"])
            ttids = batch["token_type_ids"].to(CFG["device"])
            logits = model(ids, mask, ttids)
            for k in label_keys:
                preds[k].extend(logits[k].argmax(-1).cpu().numpy())
                truth[k].extend(batch[k].numpy())
    return preds, truth

# Stage 1 report
p, t = collect_preds(r_model, r_val_loader, ["R"])
print("═"*55 + "\n  Stage 1: R\n" + "═"*55)
print(classification_report(t["R"], p["R"], target_names=R_LABELS, zero_division=0))

# Sub-model reports — unified loop over all three
for r_val, model, loader in [
    ("H", h_model, h_val_loader),
    ("M", m_model, m_val_loader),
    ("O", o_model, o_val_loader),  # added
]:
    p, t = collect_preds(model, loader, ["G","SG","CONF"])
    for col in ["G","SG","CONF"]:
        labels = SUBMODEL_VOCAB[r_val][col]
        print("═"*55 + f"\n  R={r_val} sub-model: {col}\n" + "═"*55)
        print(classification_report(t[col], p[col], target_names=labels, zero_division=0))

In [ ]:
def predict(name_district_pairs: list[tuple[str,str]], batch_size=128) -> pd.DataFrame:
    r_model.eval(); h_model.eval(); m_model.eval(); o_model.eval()  # added o_model
    texts = [f"{n.strip()} [SEP] {d.strip()}" for n, d in name_district_pairs]

    # ── Stage 1: predict R for all rows ────────────────────────────────────
    r_preds, r_confs = [], []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            enc = tokenizer(texts[i:i+batch_size], max_length=CFG["max_len"],
                            padding="max_length", truncation=True, return_tensors="pt")
            logits = r_model(enc["input_ids"].to(CFG["device"]),
                             enc["attention_mask"].to(CFG["device"]),
                             enc.get("token_type_ids", torch.zeros_like(enc["input_ids"])).to(CFG["device"]))
            probs = torch.softmax(logits["R"], -1)
            r_preds.extend(probs.argmax(-1).cpu().tolist())
            r_confs.extend(probs.max(-1).values.cpu().tolist())

    # ── Stage 2: route each row to the right sub-model ─────────────────────
    # U rows have no sub-model — they get default outputs
    h_idx = [i for i, r in enumerate(r_preds) if R_LABELS[r] == "H"]
    m_idx = [i for i, r in enumerate(r_preds) if R_LABELS[r] == "M"]
    o_idx = [i for i, r in enumerate(r_preds) if R_LABELS[r] == "O"]  # now routed to o_model
    u_idx = [i for i, r in enumerate(r_preds) if R_LABELS[r] == "U"]  # still defaults

    def run_submodel(model, vocab, indices):
        out, sub_texts = {}, [texts[i] for i in indices]
        preds = {k: [] for k in vocab}
        confs = {k: [] for k in vocab}
        with torch.no_grad():
            for j in range(0, len(sub_texts), batch_size):
                enc = tokenizer(sub_texts[j:j+batch_size], max_length=CFG["max_len"],
                                padding="max_length", truncation=True, return_tensors="pt")
                logits = model(enc["input_ids"].to(CFG["device"]),
                               enc["attention_mask"].to(CFG["device"]),
                               enc.get("token_type_ids", torch.zeros_like(enc["input_ids"])).to(CFG["device"]))
                for k in vocab:
                    probs = torch.softmax(logits[k], -1)
                    preds[k].extend(probs.argmax(-1).cpu().tolist())
                    confs[k].extend(probs.max(-1).values.cpu().tolist())
        for rank, orig_i in enumerate(indices):
            out[orig_i] = {
                "G":       vocab["G"][preds["G"][rank]],
                "G_conf":  round(confs["G"][rank], 3),
                "SG":      vocab["SG"][preds["SG"][rank]],
                "SG_conf": round(confs["SG"][rank], 3),
                "CONF":    vocab["CONF"][preds["CONF"][rank]],
            }
        return out

    sub_results = {}
    if h_idx: sub_results.update(run_submodel(h_model, SUBMODEL_VOCAB["H"], h_idx))
    if m_idx: sub_results.update(run_submodel(m_model, SUBMODEL_VOCAB["M"], m_idx))
    if o_idx: sub_results.update(run_submodel(o_model, SUBMODEL_VOCAB["O"], o_idx))  # added
    for i in u_idx:  # only true unknowns get defaults now
        sub_results[i] = {"G": "U", "G_conf": 1.0, "SG": "U", "SG_conf": 1.0, "CONF": "L"}

    # ── Assemble final dataframe ────────────────────────────────────────────
    results = []
    for i, (name, dist) in enumerate(name_district_pairs):
        results.append({
            "name": name, "district": dist,
            "R": R_LABELS[r_preds[i]], "R_conf": round(r_confs[i], 3),
            **sub_results[i]
        })
    return pd.DataFrame(results)


# ── Quick test ─────────────────────────────────────────────────────────────
predict([
    ("Swadhin Mal",    "Howrah"),
    ("Azad Mondal",      "Malda"),
    ("Pita Oraon",     "Jalpaiguri"),
    ("Mrinmoy Bhadra", "Kolkata"),
    ("Sudhin Paul",  "Murshidabad")
])